# 05 - Model Tuning, Validation & Comparison

**Objective:** Apply hyperparameter tuning, cross-validation, seed experiments, and Optuna optimization to select the best Wine classification model.

**Techniques Covered:**
- k-fold Cross Validation
- Grid Search Hyperparameter Tuning
- Random Seed Sensitivity Analysis
- Optuna Bayesian Optimization
- Model Comparison


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

import optuna
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("All imports successful")


In [ ]:
# TODO: Load the engineered data (or clean data as fallback), split into train/test, and scale
# Start by reading either engineered_data.csv or clean_data.csv from data/processed/.
# Separate features (X) from the target "class" (y).
# The Wine dataset has 13 features and a 3-class target (59/71/48 samples per class).
#
# Scale the features using StandardScaler — important for distance-based models like SVC.
# Fit on the training data only (fit_transform), then transform the test data.

PROCESSED_DIR = Path("../data/processed")

# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# X = df.drop(columns=["class"])
# y = df["class"]
#
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )
#
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
#
# print(f"Train: {X_train.shape}, Test: {X_test.shape}")
# print(f"Train class balance:\n{y_train.value_counts()}")


### Baseline Models (MLflow Tracked)

Before tuning anything, establish baseline performance with default hyperparameters.
We track each run in MLflow so we can compare results later.

Note: LogisticRegression and SVC should use the **scaled** data, while tree-based models
(RandomForest, XGBoost) use the **unscaled** data since they are invariant to feature scale.

Since Wine is a **multiclass** problem (3 cultivars), metrics like F1, precision, and recall
need `average='weighted'` to account for all classes.


In [ ]:
# Logistic Regression — simplest baseline, assumes a linear decision boundary
# For multiclass, LogisticRegression uses one-vs-rest by default.

# lr_model = LogisticRegression(max_iter=1000)
# lr_model.fit(X_train_scaled, y_train)
# y_pred_lr = lr_model.predict(X_test_scaled)
# print(f"LR Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")


In [ ]:
# TODO: Define all baseline models and log them to MLflow
# Put the models in a dictionary for easy iteration:
#   models = {
#       "LogisticRegression": LogisticRegression(max_iter=1000),
#       "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
#       "XGBoost": XGBClassifier(n_estimators=100, random_state=42),
#       "SVC": SVC(kernel="rbf", C=100),
#   }
#
# Set up an MLflow experiment named "wine_tuning".
# Then loop through each model. Inside the loop:
#   1. Start an MLflow run with mlflow.start_run(run_name=f"baseline_{name}")
#   2. Log the model type with mlflow.log_param("model_type", name)
#   3. If the model is LogisticRegression or SVC, use scaled data; otherwise use unscaled
#   4. Fit the model and make predictions
#   5. Calculate accuracy and weighted F1, log them with mlflow.log_metric()
#
# mlflow.set_experiment("wine_tuning")


### Cross-Validation Analysis

Single train/test splits can be noisy. k-fold Cross Validation splits the training data
into k folds, trains on k-1 folds and validates on the remaining fold, repeating k times.
This gives a more robust estimate of model performance.

We use `accuracy` as the scoring metric for classification.

#### Bias-Variance Connection

Cross-validation not only gives robust performance estimates — it also helps detect
**overfitting** (high variance) and **underfitting** (high bias):
- If your model scores high on the training set but much worse on CV folds, that is a
  sign of **high variance** — the model is memorizing noise in the training data.
- If both training and CV scores are poor, you likely have **high bias** — the model
  is too simple to capture the underlying pattern.
- A model with good balance will show similar train and CV scores, with the CV score
  only slightly worse than the training score.


In [ ]:
# TODO: Perform 5-fold cross-validation for the Random Forest model
# Create a KFold object with n_splits=5, shuffle=True, random_state=42.
# Then use cross_val_score() with the Random Forest model, X_train, y_train,
# and scoring="accuracy".
#
# cv = KFold(n_splits=5, shuffle=True, random_state=42)
# cv_scores = cross_val_score(
#     RandomForestClassifier(n_estimators=100, random_state=42),
#     X_train, y_train, cv=cv, scoring="accuracy"
# )
# print(f"CV Accuracy Scores: {cv_scores}")
# print(f"Mean CV Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")


In [ ]:
# TODO: Run 5-fold CV for each model
# Loop through all models and compute CV scores.
# For SVC and LogisticRegression, remember to use X_train_scaled; for tree models, use X_train.
#
# for name, model in models.items():
#     X_data = X_train_scaled if name in ["LogisticRegression", "SVC"] else X_train
#     scores = cross_val_score(model, X_data, y_train, cv=5, scoring="accuracy")
#     print(f"{name}: CV Accuracy = {scores.mean():.4f} +/- {scores.std():.4f}")


In [ ]:
# TODO: Plot cross-validation results as a box plot
# Collect CV scores for each model into a dictionary, convert to a DataFrame,
# and use sns.boxplot() to compare the distributions side by side.
# This visualizes both the central tendency and the variance of each model's CV scores.

# cv_results = {}
# for name, model in models.items():
#     X_data = X_train_scaled if name in ["LogisticRegression", "SVC"] else X_train
#     scores = cross_val_score(model, X_data, y_train, cv=5, scoring="accuracy")
#     cv_results[name] = scores
#
# results_df = pd.DataFrame(cv_results)
# sns.boxplot(data=results_df)
# plt.title("5-Fold CV Accuracy Comparison")
# plt.ylabel("Accuracy")
# plt.show()


### Grid Search Hyperparameter Tuning

Grid Search exhaustively tries every combination of hyperparameters you specify.
It is simple and thorough, but can be slow if the parameter grid is large.
We use GridSearchCV which combines grid search with k-fold cross-validation.

#### Bias-Variance Connection

Hyperparameters are the controls for the **bias-variance tradeoff**.
Each hyperparameter shifts the balance toward one end of the spectrum:
- **`C`** (LogisticRegression/SVC): lower C = stronger regularization → higher bias,
  lower variance. Higher C = less regularization → lower bias, higher variance.
- **`max_depth`** (RF/XGBoost): deeper trees capture more detail → lower bias,
  higher variance. Shallower trees → higher bias, lower variance.
- **`n_estimators`** (RF/XGBoost): more trees average out noise → lower variance,
  no significant bias change (after a minimum threshold).

When tuning, track both train and validation scores to see which direction
your model is moving on the bias-variance axis.


In [ ]:
# TODO: Define parameter grids for Random Forest and XGBoost
# Each grid is a dictionary where keys are parameter names and values are lists of options.
#
# For Random Forest:
#   n_estimators: [50, 100, 200] — more trees = more stable, slower
#   max_depth: [5, 10, None] — None means unlimited depth
#
# For XGBoost:
#   n_estimators: [50, 100] — boosting rounds
#   max_depth: [3, 5, 7] — XGBoost trees are typically shallower

# param_grids = {
#     "RandomForest": {"n_estimators": [50, 100, 200], "max_depth": [5, 10, None]},
#     "XGBoost": {"n_estimators": [50, 100], "max_depth": [3, 5, 7]},
# }


In [ ]:
# TODO: Run GridSearchCV for Random Forest
# GridSearchCV takes a model, a parameter grid, cv, and scoring.
# Use cv=3 (3-fold CV) to keep training time manageable.
#
# grid_rf = GridSearchCV(
#     RandomForestClassifier(random_state=42),
#     param_grids["RandomForest"],
#     cv=3, scoring="accuracy"
# )
# grid_rf.fit(X_train, y_train)
#
# print(f"Best RF params: {grid_rf.best_params_}")
# print(f"Best RF CV Accuracy: {grid_rf.best_score_:.4f}")


### Seed Experiments (Random State Sensitivity)

Some models (Random Forest, XGBoost) use randomness during training.
Different random seeds can produce different results. Testing multiple seeds
helps you understand whether your model is stable or sensitive to the random state.

#### Bias-Variance Connection

Seed sensitivity is a direct measure of **variance**. A model whose performance
fluctuates significantly across seeds has high variance — it is unstable and
its predictions are not reproducible in a reliable way.

- Low variance: F1 varies by <1% across seeds. The model is stable.
- High variance: F1 varies by >5% across seeds. The model needs regularization
  (reduce `max_depth`, increase `min_samples_split`, or lower `learning_rate`).

Ensemble methods (Random Forest, XGBoost) are designed to reduce variance compared
to a single decision tree, but they can still show seed sensitivity on small datasets
or when trees are allowed to grow very deep.


In [ ]:
# TODO: Test how sensitive Random Forest is to the random seed
# Try seeds like [42, 123, 456, 789, 1024, 2022].
# For each seed:
#   1. Create a RandomForestClassifier(n_estimators=100, random_state=seed)
#   2. Fit on X_train, predict on X_test
#   3. Calculate accuracy and store the result
#
# After the loop, print the mean and standard deviation of the accuracy scores.
# A low standard deviation means the model is stable across different seeds.
#
# seeds = [42, 123, 456, 789, 1024, 2022]
# seed_results = []
# for seed in seeds:
#     ....


### Optuna Hyperparameter Optimization

Optuna is a Bayesian optimization framework that is smarter than Grid Search.
Instead of trying all combinations, it uses past trial results to suggest promising
hyperparameters for the next trial. This often finds better parameters in fewer iterations.

You define an **objective function** that takes a `trial` object and uses
`trial.suggest_int()`, `trial.suggest_float()`, etc. to define the search space.
The function returns a metric, and Optuna maximizes or minimizes it.

Similarly to mlflow, Optuna runs are called `studies` and each trial is a single set of hyperparameters and its resulting metric.
The `create_study()` function initializes a new study, and `study.optimize()` runs the optimization process.
`create_study()` takes parameters:

- `direction`: "maximize" or "minimize" depending on whether you want to maximize a score (like accuracy) or minimize an error.
- `sampler`: The algorithm used to suggest hyperparameters. `TPESampler` is a common choice for Bayesian optimization.
- `pruner`: An optional component that can stop unpromising trials early to save time. `MedianPruner` is a simple choice that prunes trials that perform worse than the median of completed trials.

In [ ]:
# TODO: Define the Optuna objective function for XGBoost
# Use trial.suggest_int() for integer parameters (n_estimators, max_depth)
# and trial.suggest_float() with log=True for continuous parameters (learning_rate).
#
# Inside the function, create an XGBClassifier with the suggested params,
# run 3-fold cross-validation with scoring="accuracy",
# and return the mean score. Optuna will maximize this.
# XGBoost handles multiclass natively (uses softprob objective automatically).

# def objective_xgb(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 50, 300),
#         "max_depth": trial.suggest_int("max_depth", 3, 10),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
#     }
#     model = XGBClassifier(**params, random_state=42)
#     scores = cross_val_score(model, X_train, y_train, cv=3, scoring="accuracy")
#     return scores.mean()


In [ ]:
# TODO: Run Optuna optimization for XGBoost
# Create a study with direction="maximize" (we want to maximize accuracy).
# Run optimize() with n_trials=50 (or fewer if you want faster results).
# Print the best score and best parameters.
#
# study_xgb = optuna.create_study(direction="maximize")
# study_xgb.optimize(objective_xgb, n_trials=50)
# print(f"Best XGBoost CV Accuracy: {study_xgb.best_value:.4f}")
# print(f"Best XGBoost params: {study_xgb.best_params}")


In [ ]:
# TODO: Define and run Optuna optimization for Random Forest
# Random Forest has different parameters. The search space might include:
#   - n_estimators: 50 to 300
#   - max_depth: 3 to 20
#   - min_samples_split: 2 to 20
#
# Define the objective function similar to XGBoost, but for Random Forest.
# Then create a study and optimize it, printing the best results.
#
# print(f"Best RF CV Accuracy: {study_rf.best_value:.4f}")
# print(f"Best RF params: {study_rf.best_params}")


### Model Comparison & Saving

After all tuning is complete, compile the results into a comparison table,
select the best overall model, and save it for later use.

#### Bias-Variance Connection

When comparing baseline vs tuned models, check for signs of overfitting:
- If test accuracy improved but **train accuracy slightly decreased**, you likely reduced
  bias at an acceptable variance cost — this is healthy tuning.
- If **train accuracy is excellent** but test accuracy is poor, you **overfit** — the model
  has high variance. Increase regularization (reduce `max_depth`,
  increase `min_samples_split`, or decrease `C` for linear models).
- If **both train and test accuracy are poor**, the model has high bias — it is too
  simple. Increase model capacity (more trees, deeper trees, lower regularization).

A well-tuned model sits at the sweet spot of the bias-variance tradeoff:
low enough bias to capture the pattern, low enough variance to generalize.

Edit the Code cells in the notebook to track your experiment results. Either run them sequentially as presented, or
create a loop to run multiple seeds or hyperparameter combinations. The key is to track everything in MLflow so you can compare later!

In [ ]:
# TODO: Compile baseline and tuned results into a comparison DataFrame
# Create a table with Model names, Baseline Accuracy, and Tuned Accuracy.
# Then plot a grouped bar chart to visualize improvements from tuning.
#
# comparison = pd.DataFrame({
#     "Model": ["LogisticRegression", "RandomForest", "XGBoost", "SVC"],
#     "Baseline_Accuracy": [...],  # fill in your baseline scores
#     "Tuned_Accuracy": [...],     # fill in your tuned scores
# })
# print(comparison)
#
# comparison_melted = comparison.melt(id_vars=["Model"], var_name="Type", value_name="Accuracy")
# sns.barplot(data=comparison_melted, x="Model", y="Accuracy", hue="Type")
# plt.title("Model Performance: Baseline vs Tuned")
# plt.xticks(rotation=45)
# plt.show()


In [ ]:
# TODO: Train the best model on the full training set and save it
# Use the best parameters from Optuna (or GridSearch) to create the final model.
# Fit it on the full training data, then save with joblib.

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# best_params = study_xgb.best_params
# best_model = XGBClassifier(**best_params, random_state=42)
# best_model.fit(X_train, y_train)
# joblib.dump(best_model, MODELS_DIR / "best_tuned_model.pkl")
# print("Best model saved to models/best_tuned_model.pkl")
#
# TODO: Verify the saved model loads and produces consistent predictions
# loaded = joblib.load(MODELS_DIR / "best_tuned_model.pkl")
# y_pred = loaded.predict(X_test)
# print(f"Final model Accuracy: {accuracy_score(y_test, y_pred):.4f}")


## Data Saving

Large datasets can be saved in efficient formats like Parquet or HDF5, which support fast read/write operations and compression. This is especially useful when working with large datasets that may not fit into memory.

**Parquet** is a columnar storage file format that is optimized for performance and storage efficiency.

Saving and loading data in parquet format is made easy with pandas using the `.to_parquet()` and `.read_parquet()` methods.

In [ ]:
# TODO: Save evaluation results for later analysis
# Save the results DataFrame to a CSV or Parquet file in the processed directory.

### Exercises

1. **Add More Models**: Include `GradientBoostingClassifier` and `AdaBoostClassifier` in the comparison.
2. **Feature Selection**: Use `SelectKBest` or `RFE` to select top features before training. Does it improve performance?
3. **Ensemble Methods**: Create a VotingClassifier combining Logistic, XGBoost, and RandomForest. Compare with individual models.
4. **Bias-variance diagnosis**: Train a Random Forest with `max_depth=2` (high bias) and another with `max_depth=None` (high variance). Compute train and test accuracy for both. Which regime does your best model fall into?
5. **ROC Curves**: For multiclass, plot one-vs-rest ROC curves for each class. Which species is easiest to classify?
6. **Custom Optuna Objective**: Modify the Optuna objective to optimize weighted F1-score instead of accuracy. How do the results change?